# AI 이미지 분석 서버
FastAPI 기반 이미지 분석 API 서버 (OLLAMA / GPT 멀티모델)

In [ ]:
# ──────────────────────────────────────────────────────────────
# 패키지 설치 (최초 1회)
# pip install fastapi uvicorn ollama openai python-multipart python-dotenv
# ──────────────────────────────────────────────────────────────
import subprocess
subprocess.run([
    'pip', 'install', '-q',
    'fastapi', 'uvicorn', 'ollama', 'openai',
    'python-multipart', 'python-dotenv'
])

In [ ]:
# ──────────────────────────────────────────────────────────────
# 임포트 및 .env 설정 로드
# ──────────────────────────────────────────────────────────────
import os
import base64
import threading
import uvicorn
from dotenv import load_dotenv

from fastapi import FastAPI, File, Form, UploadFile, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import JSONResponse

import ollama
from openai import OpenAI

load_dotenv()

USE_MODEL       = os.getenv('USE_MODEL', 'OLLAMA')
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434')
OLLAMA_MODEL    = os.getenv('OLLAMA_MODEL', 'gemma4:e2b')
OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY', '')
OPENAI_MODEL    = os.getenv('OPENAI_MODEL', 'gpt-4o')

print(f'모델 설정: {USE_MODEL}')

In [ ]:
# ──────────────────────────────────────────────────────────────
# FastAPI 앱 초기화
# ──────────────────────────────────────────────────────────────
app = FastAPI(title='AI 이미지 분석 서버')

app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)

In [ ]:
# ──────────────────────────────────────────────────────────────
# 모델 호출 함수
# ──────────────────────────────────────────────────────────────

def callOllama(imageBytes: bytes, question: str) -> str:
    """ Ollama 로컬 모델로 이미지 분석 후 답변 반환 """
    response = ollama.chat(
        model=OLLAMA_MODEL,
        messages=[
            {
                'role': 'user',
                'content': question,
                'images': [imageBytes]
            }
        ]
    )
    return response['message']['content']


def callGpt(imageBytes: bytes, question: str) -> str:
    """ OpenAI GPT-4o로 이미지 분석 후 답변 반환 """
    client = OpenAI(api_key=OPENAI_API_KEY)
    b64Image = base64.b64encode(imageBytes).decode('utf-8')
    response = client.chat.completions.create(
        model=OPENAI_MODEL,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image_url',
                        'image_url': {
                            'url': f'data:image/jpeg;base64,{b64Image}'
                        }
                    },
                    {
                        'type': 'text',
                        'text': question
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content

In [ ]:
# ──────────────────────────────────────────────────────────────
# /analyze 엔드포인트
# ──────────────────────────────────────────────────────────────

@app.post('/analyze')
async def analyzeImage(
    image: UploadFile = File(...),
    question: str = Form(default='이 이미지에서 텍스트를 추출하고 내용을 설명해줘.')
):
    """ 이미지와 질문을 받아 선택된 모델로 분석 결과를 반환하는 API """
    try:
        allowedTypes = ['image/jpeg', 'image/png', 'image/webp', 'image/gif']
        if image.content_type not in allowedTypes:
            raise HTTPException(status_code=400, detail='지원하지 않는 이미지 형식입니다.')

        imageBytes = await image.read()

        if USE_MODEL == 'GPT':
            answer = callGpt(imageBytes, question)
            usedModel = OPENAI_MODEL
        elif USE_MODEL == 'OLLAMA':
            answer = callOllama(imageBytes, question)
            usedModel = OLLAMA_MODEL
        else:
            raise HTTPException(status_code=500, detail=f'알 수 없는 모델 설정: {USE_MODEL}')

        return JSONResponse(content={
            'success': True,
            'model': usedModel,
            'question': question,
            'answer': answer
        })

    except HTTPException as httpErr:
        raise httpErr
    except Exception as e:
        return JSONResponse(
            status_code=500,
            content={'success': False, 'message': str(e)}
        )


@app.get('/')
async def healthCheck():
    """ 서버 상태 및 현재 모델 설정 확인 """
    return {'success': True, 'useModel': USE_MODEL}

In [ ]:
# ──────────────────────────────────────────────────────────────
# 서버 실행 (Jupyter에서 백그라운드 스레드로 구동)
# ──────────────────────────────────────────────────────────────

def runServer():
    """ uvicorn 서버를 백그라운드 스레드에서 실행 """
    uvicorn.run(app, host='0.0.0.0', port=8000)

serverThread = threading.Thread(target=runServer, daemon=True)
serverThread.start()

print('서버 시작: http://localhost:8000')
print(f'현재 모델: {USE_MODEL}')
print('API 문서: http://localhost:8000/docs')

## 테스트 방법

```bash
curl -X POST http://localhost:8000/analyze \
  -F "image=@test.jpg" \
  -F "question=이 이미지에서 텍스트를 추출해줘"
```

또는 브라우저에서 `http://localhost:8000/docs` 접속 → Swagger UI 사용

## 모델 전환
`.env` 파일에서 `USE_MODEL=GPT` 또는 `USE_MODEL=OLLAMA` 변경 후 커널 재시작